In [1]:
from funcoes_fundamentais import *
import pandas as pd
import numpy as np


# Origem: inicial_teste.ipynb


In [ ]:
# Instalação das dependências necessárias no ambiente
# Nota: spacecore==0.4.2 é recomendado para total compatibilidade com sdplab==0.0.1
! pip install -q numpy matplotlib scipy "spacecore==0.4.2" sdplab cvxpy clarabel scs jax

# Problema das Marginais Quânticas (Quantum Marginal Problem - QMP) via SDP

O **Problema das Marginais Quânticas (Quantum Marginal Problem - QMP)** 

> **Definição:** Dado um conjunto de operadores de densidade reduzidos (marginais locais) $\{\omega_S\}_{S \in \mathcal{S}}$ definidos em subsistemas $S \subset \{1, \dots, N\}$, existe um estado quântico global $\rho$ de todo o sistema cuja redução a cada subsistema $S$ seja exatamente com $\omega_S$?
>
> $$\operatorname{Tr}_{S^c}(\rho) = \omega_S, \quad \forall S \in \mathcal{S}$$

-

### Estrutura deste Notebook

Neste notebook, abordaremos a resolução do QMP de forma modular e rigorosa:
1. **Estados Fundamentais e Projetores:** Representação de qubits e estados de Bell.
2. **Condições Físicas de Matrizes Densidade:** Hermiticidade, positividade semidefinida e traço unitário.
3. **Produto Tensorial:** Construção da família de estados GHZ.
4. **Traço Parcial:** Traço parcial via contração de índices (`np.einsum`).
5. **Base Ortonormal Hermitiana:** Base para o espaço $\mathrm{Herm}(d)$..
6. **Incorporação de Operadores Locais (`embed_operator`):** 
7. **Montagem do SDP via SDPLab (`build_qmp_sdp`):** Mapeamento do cone semidefinido.
8. **Resolução e Teste (`solve_qmp` e `verify_marginals`):** Execução do solver cônico (CLARABEL/SCS).
9. **Exemplos:**
   - **Exemplo 1 (GHZ):** Mixed = SIM, Pure = SIM.
   - **Exemplo 2 (Maximamente Misto):** Mixed = SIM, Pure = NÃO.
   - **Exemplo 3 (Incompatibilidade / Monogamia de Bell):** Mixed = NÃO, Pure = NÃO (com certificado dual de infactibilidade).
   - **Exemplo 4 (Escalabilidade e a Maldição da Dimensão):** Análise numérica com mais qubits ($N \ge 4$), crescimento exponencial do espaço de Hilbert e complexidade computacional do SDP.

In [3]:
from __future__ import annotations

import sys
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
# Bibliotecas SpaceCore e SDPLab para Programação Semidefinida 
import spacecore
from spacecore import Context, DenseVectorSpace, HermitianSpace, NumpyOps
import sdplab
from sdplab import DenseConstraintOp, SDPProblem
from sdplab.solvers import run_cvxpy_solver

np.set_printoptions(precision=4, suppress=True)

from funcoes_fundamentais import zero, one, ketbra, verifica_densidade, verify_marginals, base_hermitiana, ghz, partial_trace, tensor, embed_operator, solve_qmp, build_qmp_sdp, parse_subsystem_key, basis
print(f"Versão SDPLab:    {sdplab.__version__}")
print(f"Versão SpaceCore: {spacecore.__version__}")

Versão SDPLab:    0.0.1
Versão SpaceCore: 0.4.2


## 1. Definição de Estados e Operadores Elementares

Em computação quântica, o espaço de estados de um único qubit é o espaço de Hilbert bidimensional $\mathcal{H}_2 \cong \mathbb{C}^2$, com a base computacional ortonormal:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

Dado um vetor de estado puro normalizado $|\psi\rangle$, o operador densidade associado é o projetor ortogonal de posto 1:

$$\rho = |\psi\rangle\langle\psi|$$

A função `ketbra(psi)` calcula este produto externo $\psi \psi^\dagger$.

In [2]:
# Qubits da base computacional



phi_plus = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2.0)
rho_phi_plus = ketbra(phi_plus)

print("Vetor de Bell |Phi+>:")
print(phi_plus)
print("\nMatriz densidade rho_Phi+ = |Phi+><Phi+| (4x4):")
print(rho_phi_plus)

Vetor de Bell |Phi+>:
[0.7071+0.j 0.    +0.j 0.    +0.j 0.7071+0.j]

Matriz densidade rho_Phi+ = |Phi+><Phi+| (4x4):
[[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]]


## 2. Condições de uma Matriz Densidade


1. **Hermiticidade:** $\rho = \rho^\dagger$. 
2. **Positividade Semidefinida:** $\rho \succeq 0 \iff \forall |\phi\rangle, \langle\phi|\rho|\phi\rangle \ge 0$ $\iff$  todos os autovalores de $\rho$ são reais e não negativos: $\lambda_i(\rho) \ge 0$.
3. **Normalização (Traço Unitário):** $\operatorname{Tr}(\rho) = \sum_i \lambda_i = 1$.

A função `verifica_densidade(rho, tol)` testa computacionalmente cada um desses requisitos.

In [ ]:


resultado_verif = verifica_densidade(rho_phi_plus)
print("Auditoria do estado rho_Phi+:")
for k, v in resultado_verif.items():
    print(f"  {k:12s}: {v}")

## 3. Produto Tensorial e o Estado GHZ



A função `tensor(*args)` realiza o produto de Kronecker múltiplos vetores:


### A Família de Estados GHZ (Greenberger-Horne-Zeilinger)
Para três qubits ($N=3$), o estado GHZ parametrizado por uma fase $\theta \in [0, 2\pi)$ é definido por:

$$|GHZ_\theta\rangle = \frac{1}{\sqrt{2}} (|000\rangle + e^{i\theta} |111\rangle)$$



In [ ]:

# Exemplo: Estado |000>
psi_000 = tensor(zero, zero, zero)
print(f"Dimensão do vetor |000>: {psi_000.shape}")
#Teste com o estado GHZ


psi_ghz = ghz(theta=0.0)
rho_ghz = ketbra(psi_ghz)

print(f"Matriz densidade rho_GHZ: formato = {rho_ghz.shape}")
print("Verificação de rho_GHZ:", verifica_densidade(rho_ghz))

## 4. Traço Parcial (`np.einsum`)

Dada uma matriz densidade global $\rho_{AB}$ de um sistema bipartido, o estado do subsistema $A$ isolado é obtido eliminando-se os graus de liberdade de $B$ por meio do **traço parcial**:

$$\rho_A = \operatorname{Tr}_B(\rho_{AB}) = \sum_{j} (I_A \otimes \langle j_B|) \rho_{AB} (I_A \otimes |j_B\rangle)$$

### Implementação Geral via Tensores
Para um sistema arbitrário de $N$ corpos com dimensões locais $d_0, d_1, \dots, d_{N-1}$, a matriz densidade global $\rho$ pode ser tratada como um tensor de ordem $2N$:

$$\rho_{a_0 a_1 \dots a_{N-1},\, b_0 b_1 \dots b_{N-1}}$$

onde os índices $a_k$ correspondem aos índices "bra" (linhas) e $b_k$ aos índices "ket" (colunas).

Para traçar fora um subsistema $c \in S^c$, impõe-se a contração $a_c = b_c$ e soma-se sobre este índice. Utilizando `np.einsum`, essa operação é realizada de forma vetorizada.

In [ ]:

# Calculando as marginais de dois qubits do estado |GHZ>
rho_AB_ghz = partial_trace(rho_ghz, keep=[0, 1], dims=[2, 2, 2])
rho_AC_ghz = partial_trace(rho_ghz, keep=[0, 2], dims=[2, 2, 2])
rho_BC_ghz = partial_trace(rho_ghz, keep=[1, 2], dims=[2, 2, 2])

print("Marginal rho_AB do estado GHZ (4x4):")
print(rho_AB_ghz)
print("\nAutovalores de rho_AB:", np.round(np.linalg.eigvalsh(rho_AB_ghz), 4))

## 5. Base Ortonormal no Espaço das Matrizes Hermitianas $\mathrm{Herm}(d)$

Para formular as restrições matriciais de traço parcial $\operatorname{Tr}_{S^c}(\rho) = \omega_S$ como igualdades escalares reais $\operatorname{Tr}(M_i \rho) = b_i$, precisamos projetar a igualdade matricial sobre uma base ortonormal do espaço vetorial real das matrizes Hermitianas $\mathrm{Herm}(d_S)$.

O espaço $\mathrm{Herm}(d)$ tem dimensão real $d^2$. Sob o **produto interno de Hilbert-Schmidt**:

$$\langle A, B \rangle_{\mathrm{HS}} = \operatorname{Tr}(A^\dagger B) = \operatorname{Tr}(A B) \quad (\text{para } A, B \in \mathrm{Herm}(d))$$

uma base ortonormal $\{E_k\}_{k=1}^{d^2}$ satisfaz $\operatorname{Tr}(E_k E_l) = \delta_{kl}$ e é construída por:
1. **$d$ elementos diagonais:**
   $$E_{ii} = |i\rangle\langle i|$$
2. **$\frac{d(d-1)}{2}$ elementos fora da diagonal simétricos (reais):**
   $$E_{ij}^{\mathrm{re}} = \frac{|i\rangle\langle j| + |j\rangle\langle i|}{\sqrt{2}}, \quad (i < j)$$
3. **$\frac{d(d-1)}{2}$ elementos fora da diagonal antissimétricos (imaginários):**
   $$E_{ij}^{\mathrm{im}} = \frac{i(|i\rangle\langle j| - |j\rangle\langle i|)}{\sqrt{2}}, \quad (i < j)$$

Total de elementos: $d + 2 \times \frac{d(d-1)}{2} = d^2$ matrizes Hermitianas ortonormais.

In [ ]:


# Teste com d = 2 (um q-bit)
base_q = base_hermitiana(2)
print(f"Número de matrizes na base de Herm(2): {len(base_q)} (esperado: 2^2 = 4)")

# Teste de ortonormalidade: Matriz de Gram Tr(E_i E_j)
gram = np.array([[float(np.real(np.trace(e1 @ e2))) for e2 in base_q] for e1 in base_q])
print("Matriz de Gram Tr(E_k E_l) para d=2 (deve ser a identidade 4x4):")
print(gram)

## 6. Extendendo Operadores Locais para o Espaço Total (`embed_operator`)

Como relacionar as restrições locais sobre $\omega_S$ diretamente com o estado global $\rho$?

Pela **dualidade do traço parcial** sob o produto interno de Hilbert-Schmidt:

$$\operatorname{Tr}[ E_k \operatorname{Tr}_{S^c}(\rho) ] = \operatorname{Tr}[ (E_k \otimes I_{S^c}) \rho ]$$

Portanto, a condição matricial de consistência marginal $\operatorname{Tr}_{S^c}(\rho) = \omega_S$ decompõe-se em $d_S^2$ restrições afins escalares sobre $\rho$:

$$\operatorname{Tr}(M_{S, k} \rho) = b_{S, k}, \quad \forall k=1, \dots, d_S^2$$

onde:
- $M_{S, k} = E_k \otimes I_{S^c}$ é o observável local $E_k$ **incorporado (embedded)** no espaço global $\mathcal{H}$;
- $b_{S, k} = \operatorname{Tr}(E_k \omega_S) \in \mathbb{R}$ é o valor esperado prescrito pela marginal.

A função `embed_operator(O_s, sistema, dim)` constrói este operador global $M_{S, k}$ posicionando as matrizes de identidade exatamente nos subsistemas complementares $S^c$.

In [ ]:

# Demonstração numérica da identidade de dualidade:
# Tr[(O_A (x) I_BC) rho_GHZ] == Tr[O_A Tr_BC(rho_GHZ)]
O_teste = np.array([[2.0, 1.0 - 1.0j], [1.0 + 1.0j, 3.0]], dtype=complex)
M_global_teste = embed_operator(O_teste, sistema=[0], dim=[2, 2, 2])

val_global = np.trace(M_global_teste @ rho_ghz)
val_local = np.trace(O_teste @ partial_trace(rho_ghz, keep=[0], dims=[2, 2, 2]))

print(f"Tr[(O_A (x) I_BC) rho_GHZ] = {val_global.real:.6f} + {val_global.imag:.1e}j")
print(f"Tr[O_A Tr_BC(rho_GHZ)]     = {val_local.real:.6f} + {val_local.imag:.1e}j")
print(f"Diferença absoluta:          {abs(val_global - val_local):.2e}")

## 7. Construção do Problema SDP via SDPLab (`build_qmp_sdp`)

Com as restrições afins formuladas, podemos agora montar o problema no formato padrão cônico da biblioteca **SDPLab**:

$$\begin{aligned}
\min_{X \in \mathrm{Herm}(d)} \quad & \langle C, X \rangle = 0 \\
\text{sujeito a} \quad & \mathcal{A}(X) = b, \\
& X \succeq 0
\end{aligned}$$

onde:
- O espaço de domínio é $X = \rho \in \mathrm{Herm}(d_{\mathrm{total}})$;
- O vetor de custo $C = 0$ reflete que buscamos apenas a **factibilidade** do estado;
- O operador afim $\mathcal{A}$ empilha todas as restrições:
  1. **Normalização:** $\operatorname{Tr}(I_{d_{\mathrm{total}}} X) = 1.0$;
  2. **Consistência das Marginais:** $\operatorname{Tr}(M_{S, k} X) = b_{S, k}$.

### Detalhe do Pareamento de Frobenius no SDPLab
Na convenção interna da biblioteca `sdplab`, a ação de um operador de restrição densa $T$ sobre a matriz $X$ é dada pelo produto interno de Frobenius:

$$(\mathcal{A} X)_i = \sum_{p, q} T_{i, pq} X_{pq} = \operatorname{Tr}(T_i^T X)$$

Como desejamos medir $\operatorname{Tr}(M_i X) = b_i$ para matrizes Hermitianas $M_i$, fornecemos $T_i = M_i^T$ utilizando `np.swapaxes(M_full, -1, -2)`.

## 8. Resolução (`solve_qmp`) e Teste Numérico (`verify_marginals`)

Para resolver o problema SDP formulado, utilizamos o backend `run_cvxpy_solver` do SDPLab, integrando com o solver **CLARABEL** ou opcionalmente **SCS**.

### Diagnóstico de Factibilidade e Infactibilidade
- **Problema Factível (`optimal`):** O solver encontra uma matriz densidade global $\rho \succeq 0$ que satisfaz todas as restrições com precisão numérica de máquina ($10^{-8} \sim 10^{-15}$).
- **Problema Infactível (`infeasible`):** Pelo **Teorema da Alternativa de Farkas** para cones semidefinidos, o solver cônico gera um **certificado dual de infactibilidade**, demonstrando rigorosamente a não existência de qualquer estado global compatível. Neste caso, uma exceção é tratada e capturada de forma elegante pela função `solve_qmp`.

### Teste Numérica (`verify_marginals` / `verifica_solucao`)
Ao recuperar uma matriz $\rho$, realizamos 6 testes de integridade física:
1. **Erro de Hermiticidade:** $\|\rho - \rho^\dagger\|_F$;
2. **Erro de Normalização:** $|\operatorname{Tr}(\rho) - 1|$;
3. **Condição de Positividade:** $\lambda_{\min}(\rho) \ge -\mathrm{tol}$;
4. **Espectro Completo:** Exibição de todos os autovalores $\lambda_i$;
5. **Pureza:** $\gamma = \operatorname{Tr}(\rho^2) \in [1/d, 1]$;
6. **Resíduo das Marginais:** Erro Frobenius $\|\operatorname{Tr}_{S^c}(\rho) - \omega_S\|_F$ e erro absoluto máximo para cada subsistema.